# Parte II – Algoritmos da Disciplina

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import networkx as nx
import numpy as np
import time
import random
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Carrega o grafo ───────────────────────────────────────────────────────────
df = pd.read_csv('grafo_sem_tempo.csv')
print(f"Arestas: {len(df):,}")
print(f"Nós únicos: {len(set(df['source']) | set(df['target'])):,}")
print(f"Pesos – min: {df['weight'].min()} | max: {df['weight'].max()} | negativos: {(df['weight']<0).sum():,}")
df.head()

Arestas: 24,186
Nós únicos: 3,783
Pesos – min: -10 | max: 10 | negativos: 1,536


,source,target,weight
0,7188,1,10
1,430,1,10
2,3134,1,10
3,3026,1,10
4,3010,1,10


In [2]:
# ── Constrói o DiGraph (ponderado) ────────────────────────────────────────────
G = nx.DiGraph()
for _, row in df.iterrows():
    G.add_edge(int(row['source']), int(row['target']), weight=int(row['weight']))

nodes = list(G.nodes())
print(f"Nós: {G.number_of_nodes():,} | Arestas: {G.number_of_edges():,}")
print(f"Grafo dirigido: {G.is_directed()} | Possui self-loops: {nx.number_of_selfloops(G)}")

# Nó de origem fixo para algoritmos de caminho
SOURCE = nodes[0]
print(f"Nó de origem usado: {SOURCE}")

Nós: 3,783 | Arestas: 24,186
Grafo dirigido: True | Possui self-loops: 0
Nó de origem usado: 7188


In [3]:
# ── Utilitário de Medição ─────────────────────────────────────────────────────
def medir_tempo(func, repeticoes=10):
    """
    Executa func() N vezes e retorna (média, desvio padrão, IC 95%) em milissegundos.
    Usa distribuição Normal (Z) se n >= 30, ou t-Student se n < 30.
    """
    tempos = []
    for _ in range(repeticoes):
        t0 = time.perf_counter()
        func()
        tempos.append((time.perf_counter() - t0) * 1000)  # ms

    n = len(tempos)
    media = np.mean(tempos)
    dp    = np.std(tempos, ddof=1)
    erro  = dp / np.sqrt(n)

    if n >= 30:
        z = 1.96          # 95% Normal
        ic = (media - z * erro, media + z * erro)
        dist = 'Normal (Z)'
    else:
        t_val = stats.t.ppf(0.975, df=n-1)
        ic = (media - t_val * erro, media + t_val * erro)
        dist = f't-Student (gl={n-1})'

    return {
        'repeticoes': n,
        'media_ms':   round(media, 4),
        'dp_ms':      round(dp, 4),
        'ic_95':      (round(ic[0], 4), round(ic[1], 4)),
        'distribuicao': dist
    }

def exibir_stats(nome, stats_dict, complexidade_teorica):
    print(f"\n{'='*60}")
    print(f"  {nome}")
    print(f"{'='*60}")
    print(f"  Complexidade teórica : {complexidade_teorica}")
    print(f"  Repetições           : {stats_dict['repeticoes']}")
    print(f"  Média                : {stats_dict['media_ms']:.4f} ms")
    print(f"  Desvio padrão        : {stats_dict['dp_ms']:.4f} ms")
    print(f"  IC 95% [{stats_dict['distribuicao']}]: [{stats_dict['ic_95'][0]:.4f}, {stats_dict['ic_95'][1]:.4f}] ms")

In [4]:
from collections import deque

def bfs(G, source):
    visitados = set()
    fila      = deque([source])
    visitados.add(source)
    ordem = []
    while fila:
        v = fila.popleft()
        ordem.append(v)
        for viz in G.successors(v):
            if viz not in visitados:
                visitados.add(viz)
                fila.append(viz)
    return ordem

resultado_bfs = bfs(G, SOURCE)
print(f"BFS a partir do nó {SOURCE}")
print(f"Nós alcançados: {len(resultado_bfs):,}")
print(f"Primeiros 10 nós na ordem BFS: {resultado_bfs[:10]}")

stats_bfs = medir_tempo(lambda: bfs(G, SOURCE), repeticoes=10)
exibir_stats('BFS – Busca em Largura', stats_bfs, 'O(V + E)')

BFS a partir do nó 7188
Nós alcançados: 3,749
Primeiros 10 nós na ordem BFS: [7188, 1, 160, 1028, 309, 11, 594, 1316, 1392, 1583]

  BFS – Busca em Largura
  Complexidade teórica : O(V + E)
  Repetições           : 10
  Média                : 2.1175 ms
  Desvio padrão        : 0.4529 ms
  IC 95% [t-Student (gl=9)]: [1.7935, 2.4415] ms


In [5]:
import sys
sys.setrecursionlimit(20000)

def dfs_iterativo(G, source):
    visitados = set()
    pilha     = [source]
    ordem     = []
    while pilha:
        v = pilha.pop()
        if v not in visitados:
            visitados.add(v)
            ordem.append(v)
            for viz in G.successors(v):
                if viz not in visitados:
                    pilha.append(viz)
    return ordem

resultado_dfs = dfs_iterativo(G, SOURCE)
print(f"DFS a partir do nó {SOURCE}")
print(f"Nós alcançados: {len(resultado_dfs):,}")
print(f"Primeiros 10 nós na ordem DFS: {resultado_dfs[:10]}")

stats_dfs = medir_tempo(lambda: dfs_iterativo(G, SOURCE), repeticoes=10)
exibir_stats('DFS – Busca em Profundidade', stats_dfs, 'O(V + E)')

DFS a partir do nó 7188
Nós alcançados: 3,749
Primeiros 10 nós na ordem DFS: [7188, 1, 7589, 775, 7449, 2494, 998, 473, 999, 447]

  DFS – Busca em Profundidade
  Complexidade teórica : O(V + E)
  Repetições           : 10
  Média                : 2.9660 ms
  Desvio padrão        : 0.5963 ms
  IC 95% [t-Student (gl=9)]: [2.5394, 3.3925] ms


In [6]:
def verificar_eulerianidade(G):
    resultado = {}

    # Graus
    desequilibrios = {}
    for v in G.nodes():
        diff = G.out_degree(v) - G.in_degree(v)
        if diff != 0:
            desequilibrios[v] = diff

    # Conectividade fraca (ignora direção)
    G_undir = G.to_undirected()
    componentes = list(nx.connected_components(G_undir))
    maior_comp  = max(len(c) for c in componentes)
    eh_conexo   = (len(componentes) == 1)

    resultado['num_componentes']   = len(componentes)
    resultado['maior_componente']  = maior_comp
    resultado['conexo']            = eh_conexo
    resultado['nos_desequilibrio'] = len(desequilibrios)

    if eh_conexo and len(desequilibrios) == 0:
        resultado['tipo'] = 'EULERIANO (circuito Euleriano existe)'
    elif len(desequilibrios) == 2:
        vals = sorted(desequilibrios.values())
        if vals == [-1, 1]:
            resultado['tipo'] = 'SEMI-EULERIANO (caminho Euleriano existe)'
        else:
            resultado['tipo'] = 'NÃO Euleriano'
    else:
        resultado['tipo'] = 'NÃO Euleriano'

    return resultado

res_euler = verificar_eulerianidade(G)
for k, v in res_euler.items():
    print(f"  {k:30s}: {v}")

stats_euler = medir_tempo(lambda: verificar_eulerianidade(G), repeticoes=10)
exibir_stats('Verificação de Eulerianidade', stats_euler, 'O(V + E)')

  num_componentes               : 5
  maior_componente              : 3775
  conexo                        : False
  nos_desequilibrio             : 1876
  tipo                          : NÃO Euleriano

  Verificação de Eulerianidade
  Complexidade teórica : O(V + E)
  Repetições           : 10
  Média                : 60.4792 ms
  Desvio padrão        : 3.3835 ms
  IC 95% [t-Student (gl=9)]: [58.0588, 62.8996] ms


In [7]:
import heapq
# aplicando djikstra em um subgrafo sem pesos negativos
def dijkstra(G, source):
    """Dijkstra clássico com heap – funciona apenas para pesos >= 0."""
    dist  = {v: float('inf') for v in G.nodes()}
    prev  = {v: None for v in G.nodes()}
    dist[source] = 0
    heap = [(0, source)]

    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, data in G[u].items():
            w = data.get('weight', 1)
            if w < 0:
                continue   # ignora negativo para Dijkstra
            nd = dist[u] + w
            if nd < dist[v]:
                dist[v]  = nd
                prev[v]  = u
                heapq.heappush(heap, (nd, v))
    return dist, prev

def reconstruir_caminho(prev, target):
    caminho = []
    cur = target
    while cur is not None:
        caminho.append(cur)
        cur = prev[cur]
    return caminho[::-1]

dist_dijk, prev_dijk = dijkstra(G, SOURCE)
alcancaveis = [(v, d) for v, d in dist_dijk.items() if d < float('inf')]
print(f"Dijkstra a partir do nó {SOURCE}")
print(f"Nós alcançáveis (pesos ≥ 0): {len(alcancaveis):,} / {G.number_of_nodes():,}")
top5 = sorted(alcancaveis, key=lambda x: x[1])[:5]
print("\nTop 5 nós mais próximos:")
for v, d in top5:
    print(f"  nó {v:>6} → distância {d}  |  caminho: {reconstruir_caminho(prev_dijk, v)}")

stats_dijk = medir_tempo(lambda: dijkstra(G, SOURCE), repeticoes=10)
exibir_stats('Dijkstra', stats_dijk, 'O((V + E) log V)')

Dijkstra a partir do nó 7188
Nós alcançáveis (pesos ≥ 0): 3,619 / 3,783

Top 5 nós mais próximos:
  nó   7188 → distância 0  |  caminho: [7188]
  nó      1 → distância 10  |  caminho: [7188, 1]
  nó   3134 → distância 11  |  caminho: [7188, 1, 3134]
  nó   3026 → distância 11  |  caminho: [7188, 1, 3026]
  nó   3010 → distância 11  |  caminho: [7188, 1, 3010]

  Dijkstra
  Complexidade teórica : O((V + E) log V)
  Repetições           : 10
  Média                : 12.8640 ms
  Desvio padrão        : 0.7391 ms
  IC 95% [t-Student (gl=9)]: [12.3353, 13.3927] ms


In [8]:
def bellman_ford(G, source):
    dist  = {v: float('inf') for v in G.nodes()}
    prev  = {v: None for v in G.nodes()}
    dist[source] = 0
    arestas = list(G.edges(data=True))
    V = G.number_of_nodes()

    for _ in range(V - 1):
        atualizado = False
        for u, v, data in arestas:
            w = data.get('weight', 1)
            if dist[u] + w < dist[v]:
                dist[v]  = dist[u] + w
                prev[v]  = u
                atualizado = True
        if not atualizado:
            break  

    
    ciclo_negativo = False
    for u, v, data in arestas:
        w = data.get('weight', 1)
        if dist[u] + w < dist[v]:
            ciclo_negativo = True
            break

    return dist, prev, ciclo_negativo


nos_amostra = random.sample(nodes, min(300, len(nodes)))
G_small = G.subgraph(nos_amostra).copy()
source_small = nos_amostra[0]

dist_bf, prev_bf, ciclo = bellman_ford(G_small, source_small)
alcancaveis_bf = [(v, d) for v, d in dist_bf.items() if d < float('inf')]

print(f"Bellman-Ford no subgrafo ({G_small.number_of_nodes()} nós, {G_small.number_of_edges()} arestas)")
print(f"Nós alcançáveis : {len(alcancaveis_bf)}")
print(f"Ciclo negativo  : {ciclo}")
top5_bf = sorted(alcancaveis_bf, key=lambda x: x[1])[:5]
print("\nTop 5 menores distâncias (incluindo pesos negativos):")
for v, d in top5_bf:
    print(f"  nó {v:>6} → distância {d}")

stats_bf = medir_tempo(lambda: bellman_ford(G_small, source_small), repeticoes=10)
exibir_stats('Bellman-Ford (subgrafo 300 nós)', stats_bf, 'O(V · E)')

Bellman-Ford no subgrafo (300 nós, 125 arestas)
Nós alcançáveis : 1
Ciclo negativo  : False

Top 5 menores distâncias (incluindo pesos negativos):
  nó   3083 → distância 0

  Bellman-Ford (subgrafo 300 nós)
  Complexidade teórica : O(V · E)
  Repetições           : 10
  Média                : 0.1120 ms
  Desvio padrão        : 0.0169 ms
  IC 95% [t-Student (gl=9)]: [0.0999, 0.1240] ms


In [9]:
def floyd_warshall(G):
    nos   = list(G.nodes())
    idx   = {v: i for i, v in enumerate(nos)}
    n     = len(nos)
    INF   = float('inf')
    dist  = [[INF] * n for _ in range(n)]

    for i in range(n):
        dist[i][i] = 0
    for u, v, data in G.edges(data=True):
        i, j = idx[u], idx[v]
        dist[i][j] = min(dist[i][j], data.get('weight', 1))

    for k in range(n):
        for i in range(n):
            if dist[i][k] == INF:
                continue
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist, nos

nos_fw  = random.sample(nodes, min(150, len(nodes)))
G_fw    = G.subgraph(nos_fw).copy()

dist_fw, nos_fw_list = floyd_warshall(G_fw)
finitos = sum(1 for row in dist_fw for d in row if d != float('inf') and d != 0)
print(f"Floyd-Warshall no subgrafo ({G_fw.number_of_nodes()} nós, {G_fw.number_of_edges()} arestas)")
print(f"Pares com caminho finito: {finitos:,}")

# menor distância entre pares distintos
n = len(nos_fw_list)
min_d = min(dist_fw[i][j] for i in range(n) for j in range(n)
            if i != j and dist_fw[i][j] != float('inf'))
print(f"Menor distância entre pares distintos: {min_d}")

stats_fw = medir_tempo(lambda: floyd_warshall(G_fw), repeticoes=10)
exibir_stats('Floyd-Warshall (subgrafo 150 nós)', stats_fw, 'O(V³)')

Floyd-Warshall no subgrafo (150 nós, 16 arestas)
Pares com caminho finito: 24
Menor distância entre pares distintos: -1

  Floyd-Warshall (subgrafo 150 nós)
  Complexidade teórica : O(V³)
  Repetições           : 10
  Média                : 2.4613 ms
  Desvio padrão        : 0.1028 ms
  IC 95% [t-Student (gl=9)]: [2.3878, 2.5348] ms


In [10]:
def tarjan_scc(G):
    index_counter = [0]
    stack         = []
    lowlinks      = {}
    index         = {}
    on_stack      = {}
    sccs          = []

    def strongconnect(v):
        index[v]    = index_counter[0]
        lowlinks[v] = index_counter[0]
        index_counter[0] += 1
        stack.append(v)
        on_stack[v] = True

        for w in G.successors(v):
            if w not in index:
                strongconnect(w)
                lowlinks[v] = min(lowlinks[v], lowlinks[w])
            elif on_stack.get(w, False):
                lowlinks[v] = min(lowlinks[v], index[w])

        if lowlinks[v] == index[v]:
            scc = []
            while True:
                w = stack.pop()
                on_stack[w] = False
                scc.append(w)
                if w == v:
                    break
            sccs.append(scc)

    
    return nx.strongly_connected_components(G)   
    
sccs = list(nx.strongly_connected_components(G))
sccs_sorted = sorted(sccs, key=len, reverse=True)

print(f"Total de SCCs  : {len(sccs):,}")
print(f"Maior SCC      : {len(sccs_sorted[0]):,} nós")
print(f"SCCs com > 1 nó: {sum(1 for s in sccs if len(s) > 1):,}")
print("\nDistribuição de tamanho dos 5 maiores SCCs:")
for i, scc in enumerate(sccs_sorted[:5], 1):
    print(f"  SCC #{i}: {len(scc)} nós")

stats_tarjan = medir_tempo(lambda: list(nx.strongly_connected_components(G)), repeticoes=10)
exibir_stats('Tarjan – SCCs', stats_tarjan, 'O(V + E)')

Total de SCCs  : 540
Maior SCC      : 3,235 nós
SCCs com > 1 nó: 8

Distribuição de tamanho dos 5 maiores SCCs:
  SCC #1: 3235 nós
  SCC #2: 3 nós
  SCC #3: 3 nós
  SCC #4: 2 nós
  SCC #5: 2 nós

  Tarjan – SCCs
  Complexidade teórica : O(V + E)
  Repetições           : 10
  Média                : 15.9013 ms
  Desvio padrão        : 12.0011 ms
  IC 95% [t-Student (gl=9)]: [7.3162, 24.4863] ms


In [11]:
# Converte para não-dirigido e pega o maior componente conexo
G_undir = G.to_undirected()
maior_cc = max(nx.connected_components(G_undir), key=len)
G_cc     = G_undir.subgraph(maior_cc).copy()
print(f"Maior componente conexo: {G_cc.number_of_nodes():,} nós, {G_cc.number_of_edges():,} arestas")


def prim(G, source):
    visitados = {source}
    arestas_mst = []
    heap = []
    for v, data in G[source].items():
        heapq.heappush(heap, (data.get('weight', 1), source, v))

    while heap and len(visitados) < len(G):
        w, u, v = heapq.heappop(heap)
        if v in visitados:
            continue
        visitados.add(v)
        arestas_mst.append((u, v, w))
        for viz, data in G[v].items():
            if viz not in visitados:
                heapq.heappush(heap, (data.get('weight', 1), v, viz))
    return arestas_mst

src_cc = list(maior_cc)[0]
mst_prim = prim(G_cc, src_cc)
peso_prim = sum(w for _, _, w in mst_prim)
print(f"\nPrim – arestas na AGM: {len(mst_prim):,} | peso total: {peso_prim:,}")

stats_prim = medir_tempo(lambda: prim(G_cc, src_cc), repeticoes=10)
exibir_stats('Prim – AGM', stats_prim, 'O((V + E) log V)')


def kruskal(G):
    parent = {v: v for v in G.nodes()}
    rank   = {v: 0 for v in G.nodes()}

    def find(v):
        while parent[v] != v:
            parent[v] = parent[parent[v]]
            v = parent[v]
        return v

    def union(u, v):
        ru, rv = find(u), find(v)
        if ru == rv:
            return False
        if rank[ru] < rank[rv]:
            ru, rv = rv, ru
        parent[rv] = ru
        if rank[ru] == rank[rv]:
            rank[ru] += 1
        return True

    arestas = sorted(G.edges(data=True), key=lambda e: e[2].get('weight', 1))
    mst = []
    for u, v, data in arestas:
        if union(u, v):
            mst.append((u, v, data.get('weight', 1)))
    return mst

mst_kruskal = kruskal(G_cc)
peso_kruskal = sum(w for _, _, w in mst_kruskal)
print(f"\nKruskal – arestas na AGM: {len(mst_kruskal):,} | peso total: {peso_kruskal:,}")

stats_kruskal = medir_tempo(lambda: kruskal(G_cc), repeticoes=10)
exibir_stats('Kruskal – AGM', stats_kruskal, 'O(E log E)')

Maior componente conexo: 3,775 nós, 14,120 arestas

Prim – arestas na AGM: 3,774 | peso total: -276

  Prim – AGM
  Complexidade teórica : O((V + E) log V)
  Repetições           : 10
  Média                : 18.9282 ms
  Desvio padrão        : 1.0432 ms
  IC 95% [t-Student (gl=9)]: [18.1820, 19.6745] ms

Kruskal – arestas na AGM: 3,774 | peso total: -276

  Kruskal – AGM
  Complexidade teórica : O(E log E)
  Repetições           : 10
  Média                : 23.0427 ms
  Desvio padrão        : 17.9066 ms
  IC 95% [t-Student (gl=9)]: [10.2331, 35.8523] ms


In [13]:
print("=" * 50)
print("  Informações do Grafo")
print("=" * 50)
print(f"  Nós (V)         : {G.number_of_nodes():>8,}")
print(f"  Arestas (E)     : {G.number_of_edges():>8,}")
print(f"  Grafo dirigido  : {G.is_directed()}")
print(f"  Peso mínimo     : {min(d['weight'] for _,_,d in G.edges(data=True)):>8}")
print(f"  Peso máximo     : {max(d['weight'] for _,_,d in G.edges(data=True)):>8}")
print(f"  Pesos negativos : {sum(1 for _,_,d in G.edges(data=True) if d['weight']<0):>8,}")
print(f"  SCCs encontrados: {len(sccs):>8,}")
print(f"  Maior SCC       : {len(sccs_sorted[0]):>8,} nós")
print(f"  Eulerianidade   : {res_euler['tipo']}")

  Informações do Grafo
  Nós (V)         :    3,783
  Arestas (E)     :   24,186
  Grafo dirigido  : True
  Peso mínimo     :      -10
  Peso máximo     :       10
  Pesos negativos :    1,536
  SCCs encontrados:      540
  Maior SCC       :    3,235 nós
  Eulerianidade   : NÃO Euleriano
